# Planetary Computer Sentinel-1 RTC -- Coverage Check

Michel/Tom's suggestion (2026-08-30): Microsoft's Planetary Computer hosts a
`sentinel-1-rtc` collection -- gamma-nought backscatter, radiometrically
terrain-corrected with a real 10m DEM, delivered as cloud-optimized GeoTIFFs
readable via windowed HTTPS reads (no multi-GB SAFE downloads, no local
calibration). This replaces the current raw-SAFE calibration path
(`build_products()` in notebook 01/03), which does sigma-nought calibration
and GCP warping but has no DEM-based terrain correction.

**This notebook only answers one question**: does the `sentinel-1-rtc`
collection actually have coverage for Tuktoyaktuk in the LiDAR survey date
window, and can we actually read a windowed patch from it? It does not do
full patch extraction -- that's the next step if this checks out (see the
closing markdown cell).

Prerequisites before running:
- A free Planetary Computer account + API key
  (https://planetarycomputer.microsoft.com/account/request), set as
  `PC_SDK_SUBSCRIPTION_KEY` in `.env` (placeholder already exists there).
- `pip install pystac-client planetary-computer` (already in
  `requirements.txt`).

In [1]:
import os
import datetime as dt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import rasterio
from rasterio.warp import transform_geom
from shapely.geometry import box, shape
from shapely.ops import unary_union

import pystac_client
import planetary_computer
from dotenv import load_dotenv

load_dotenv()
if os.environ.get('PC_SDK_SUBSCRIPTION_KEY'):
    planetary_computer.settings.set_subscription_key(os.environ['PC_SDK_SUBSCRIPTION_KEY'])


## Configuration

Same conventions as notebook 01 -- reuses the same `REPO_DIR`, `REGION`,
`LIDAR_DIR`, and `DATE_BY_REGION` so the AOI and date window are directly
comparable to the existing CDSE raw-SAFE acquisition.

In [2]:
REPO_DIR = Path('/cs/student/project_msc/2025/aibh/jiayiche')
INPUT_DIR = REPO_DIR / 'input_data'
REGION = 'tuk'  # Tuktoyaktuk only for now -- Pond Inlet/Cambridge Bay are EW-mode sites,
                # and this collection looks IW-only (see markdown above); confirm separately
                # before assuming either way if extending scope later.
LIDAR_DIR = INPUT_DIR / f'lidar_patches_{REGION}'
DATE_BY_REGION = {
    'pondinlet': dt.date(2024, 4, 26),
    'cambridge': dt.date(2024, 4, 18),
    'tuk': dt.date(2024, 4, 16),
}
SEARCH_DAYS = 30


## Build the search AOI

Identical to notebook 01's `aoi_from_lidar_patches` -- reused verbatim so the
two acquisition paths (CDSE raw-SAFE vs. Planetary Computer RTC) query the
exact same ground footprint, making any coverage difference attributable to
the data source, not a different AOI.

In [3]:
def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if not paths:
        raise FileNotFoundError(f'No LiDAR patches found in {patches_dir}')
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]

    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
aoi_ll = aoi.convex_hull
print(f'AOI centroid (lon, lat): {aoi_ll.centroid.x:.4f}, {aoi_ll.centroid.y:.4f}')


AOI centroid (lon, lat): -133.3174, 69.7721


## Search Planetary Computer for `sentinel-1-rtc` coverage

Same ±`SEARCH_DAYS` window as the existing CDSE query, so scene counts are
directly comparable.

In [4]:
catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)

start = DATE_BY_REGION[REGION] - dt.timedelta(days=SEARCH_DAYS)
end = DATE_BY_REGION[REGION] + dt.timedelta(days=SEARCH_DAYS)

search = catalog.search(
    collections=['sentinel-1-rtc'],
    intersects=aoi_ll.__geo_interface__,
    datetime=f'{start.isoformat()}/{end.isoformat()}',
)
items = list(search.items())
print(f'Found {len(items)} scenes for {REGION} between {start} and {end}')
for item in items:
    props = item.properties
    print(f"  {item.id} | {props.get('datetime')} | mode={props.get('sar:instrument_mode')} | "
          f"orbit={props.get('sat:orbit_state')} | assets={list(item.assets.keys())}")


Found 7 scenes for tuk between 2024-03-17 and 2024-05-16
  S1A_IW_GRDH_1SDV_20240507T020856_20240507T020920_053759_06883A_rtc | 2024-05-07T02:09:08.528047Z | mode=IW | orbit=ascending | assets=['vh', 'vv', 'tilejson', 'rendered_preview']
  S1A_IW_GRDH_1SDV_20240425T020855_20240425T020920_053584_068151_rtc | 2024-04-25T02:09:07.987154Z | mode=IW | orbit=ascending | assets=['vh', 'vv', 'tilejson', 'rendered_preview']
  S1A_IW_GRDH_1SDV_20240423T022517_20240423T022541_053555_068037_rtc | 2024-04-23T02:25:29.394092Z | mode=IW | orbit=ascending | assets=['vh', 'vv', 'tilejson', 'rendered_preview']
  S1A_IW_GRDH_1SDV_20240413T020854_20240413T020918_053409_067A75_rtc | 2024-04-13T02:09:06.394434Z | mode=IW | orbit=ascending | assets=['vh', 'vv', 'tilejson', 'rendered_preview']
  S1A_IW_GRDH_1SDV_20240401T020855_20240401T020919_053234_067390_rtc | 2024-04-01T02:09:07.777942Z | mode=IW | orbit=ascending | assets=['vh', 'vv', 'tilejson', 'rendered_preview']
  S1A_IW_GRDH_1SDV_20240320T020855_202

## Confirm a windowed read actually works

If scenes were found above, this reads a small window from the first
scene's VV asset directly over HTTPS -- no download -- to confirm the
cloud-optimized-GeoTIFF read path is genuinely working end to end, not just
that the catalogue search returned results.

In [6]:
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds

if items:
    item = items[0]
    with rasterio.open(item.assets['vv'].href) as src:
        print('CRS:', src.crs)
        print('Transform:', src.transform)
        print('Full shape:', src.shape)

        aoi_bounds_scene_crs = transform_bounds('EPSG:4326', src.crs, *aoi_ll.bounds)
        window = from_bounds(*aoi_bounds_scene_crs, transform=src.transform)
        sample = src.read(1, window=window)
        print('AOI window shape:', sample.shape, '| dtype:', sample.dtype,
              '| min/max:', sample.min(), sample.max())
        print('Fraction nodata (-32768):', (sample == -32768).mean())
else:
    print('No scenes found -- nothing to read.')


CRS: EPSG:32608
Transform: | 10.00, 0.00, 474000.00|
| 0.00,-10.00, 7911420.00|
| 0.00, 0.00, 1.00|
Full shape: (23076, 30089)
AOI window shape: (1722, 380) | dtype: float32 | min/max: 0.0013574249 0.27059415
Fraction nodata (-32768): 0.0


## Interpreting the result

- **If scenes were found and the windowed read worked**: coverage is
  confirmed for Tuktoyaktuk in this date window. Next step (not in this
  notebook) would be building a patch-extraction path analogous to
  notebooks 02/03 -- but reading windowed COG data per LiDAR patch footprint
  instead of calibrating/warping a downloaded SAFE archive -- and a new
  training notebook analogous to 04/08/12 using that data.
- **If zero scenes were found**: either the date window needs adjusting, or
  this collection genuinely doesn't cover this AOI/date combination --
  worth a broader date search (e.g. remove the `datetime` filter entirely
  and check what's available for this AOI at all) before concluding it's
  unusable.
- Either way, report this result back along with the reply to Michel/Tom's
  email -- this was flagged as an open unknown in `QUESTIONS_FOR_MICHEL.md`
  (item 6) and `CONCEPTS.md`'s Planetary Computer section, and this
  notebook is what resolves it.